# Import packages and load data

In [ ]:
import pandas as pd
import numpy as np
import time
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.gaussian_process.kernels import (
    RBF, WhiteKernel, ConstantKernel as C, DotProduct
)
from joblib import dump, load
import matplotlib.pyplot as plt
import mplcursors
import plotly.express as px
import os
from sklearn.exceptions import ConvergenceWarning
import warnings
warnings.filterwarnings("ignore", message="X has feature names, but GaussianProcessRegressor was fitted without feature names")
warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [ ]:
file_path = '../chemical-featurization/likeness_data/poly_maccs.xlsx'
model_output = 'gisaxs_domain'
# import data frame for pre-processing
df = pd.read_excel(file_path)

In [ ]:
# Drop rows with any NaN values
df_clean = df.dropna()

# Assign input variables and target variable
inputs = df_clean.iloc[:, 1:-2].copy()
target = df_clean[f'{model_output}']

# GPR training

In [ ]:
%%time

X_train, X_test, y_train, y_test = train_test_split(
    inputs, target, test_size=0.2, random_state=10
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

n_features = X_train_scaled.shape[1]

kernel = (

    C(1.0, (1e-3, 1e3)) * DotProduct(sigma_0=1.0)
    # + C(1.0, (1e-2, 1e2)) * RBF(
        # length_scale=np.ones(n_features),
        # length_scale_bounds=(1e-2, 50.0)
    )
    + WhiteKernel(
        noise_level=1e-3,
        # noise_level_bounds=(1e-6, 1e-1)
    )
)

gpr = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    # alpha=1e-10,
    # random_state=42
)

gpr.fit(X_train, y_train)

y_pred, y_std = gpr.predict(X_test, return_std=True)
test_r2 = r2_score(y_test, y_pred)

print("\nLearned Kernel:")
print(gpr.kernel_)
print(f"\nTest R2: {test_r2}")
dump(gpr , f'raw_models/{model_output}_gpr.joblib')

In [ ]:
validation_path = '../chemical-featurization/likeness_data/validation/poly_maccs_validation_subset.xlsx'
# import data frame for pre-processing
df_val = pd.read_excel(validation_path)

In [ ]:
# Drop missing values in validation set
df_val_clean = df_val.dropna()
sample_names = df_val_clean["Sample"]
# Assign inputs and target (must match training exactly)
X_val = df_val_clean[X_train.columns]  # enforce same columns & order as training
y_val = df_val_clean[f'{output}']

# Scale using TRAINING scaler (no refitting)
X_val_scaled = scaler.transform(X_val)

# Predictions
y_val_pred, y_val_std = gpr.predict(X_val, return_std=True)

# Metrics
val_r2 = gpr.score(X_val, y_val)
val_mae = mean_absolute_error(y_val, y_val_pred)

print(f'Validation R2: {val_r2}, validation mae: {val_mae}')

In [ ]:
plot_df = pd.DataFrame({
    "True": y_val.values,
    "Predicted": y_val_pred,
    "StdDev": y_val_std,
    "Sample": sample_names
})

fig = px.scatter(
    plot_df,
    x="True",
    y="Predicted",
    hover_name="Sample",
    hover_data={"True": True, "Predicted": True, "StdDev": True},
    title=f"Parity Plot: {output}"
)

min_y = plot_df[["True", "Predicted"]].min().min()
max_y = plot_df[["True", "Predicted"]].max().max()

fig.add_shape(
    type="line",
    x0=min_y, y0=min_y,
    x1=max_y, y1=max_y,
    line=dict(dash="dash")
)

fig.update_layout(
    xaxis_title="True Values",
    yaxis_title="Predicted Values",
    width=600,
    height=600
)

fig.show()

In [ ]:
out_dir = "likeness_validation_scatter"
os.makedirs(out_dir, exist_ok=True)


val_errors = np.abs(y_val - y_val_pred)

df_clean = df.dropna()
df_val_clean = df_val.dropna()

X = df_clean.iloc[:, 1:50].copy()
X_val = df_val_clean.iloc[:, 1:50].copy()

X_train, X_test = train_test_split(
    X, test_size=0.2, random_state=10
)
likeness_features = [
    col for col in X.columns if "likeness" in col.lower()
]

for feature in likeness_features:
    # Training values
    train_vals = X_train[feature].dropna().values
    train_min = np.min(train_vals)
    train_max = np.max(train_vals)
    p_low, p_high = np.percentile(train_vals, [1, 99])

    # Validation values and corresponding errors
    val_vals = X_val[feature].dropna()
    val_indices = val_vals.index
    errors = val_errors.loc[val_indices]

    # Small vertical jitter to separate overlapping points
    y_jitter = np.linspace(-0.3, 0.3, len(val_vals))

    plt.figure(figsize=(7, 4))

    # Scatter validation points colored by error
    sc = plt.scatter(
        val_vals,
        y_jitter,
        c=errors,
        cmap="plasma",
        s=80,
        edgecolor="k",
        alpha=0.8
    )

    # Training full range
    plt.axvline(train_min, linestyle="--", color="black", alpha=0.6, label="Train min/max")
    plt.axvline(train_max, linestyle="--", color="black", alpha=0.6)

    plt.xlabel(feature)
    plt.ylabel("Jittered samples")
    plt.title(f"{feature} | Validation Error Scatter")
    plt.colorbar(sc, label="Absolute Prediction Error")
    plt.legend()
    plt.tight_layout()
    # plt.savefig(os.path.join(out_dir, f"{feature}_ValidationErrorScatter.png"), dpi=300)
    plt.show()